### 1. 지출, 예산 관리 클래스 생성
- 내부 유틸 함수와 외부로 사용할 함수 따로 만듦
- 지출, 예산 기본 로직 같음 (등록,수정,삭제,검색)
- 같은 클래스로 만들어 인스턴스를 따로 생성하여 이용
    - budget = Ledger("예산")
    - expenditure = Ledger("지출")

In [4]:
"""예산/지출 등 거래 내역을 관리하는 공용 원장(ledger).
등록/수정/삭제 이력을 전부 로그로 남기고, 현재값은 로그 합산으로 계산한다.
거래는 (date, tag)가 아니라 register()마다 발급되는 고유 id로 식별된다
(같은 날 같은 태그로 두 번 등록해도 서로 다른 거래로 구분됨).
"""

from datetime import date as _date


class Ledger:
    def __init__(self, label="내역"):
        self.records = []  # [{id, date, tag, amount, action}, ...] 전체 이력 (append-only)
        self._next_id = 0
        self.label = label  # 에러 메시지에 쓰이는 이름 (예: "예산", "지출")

    # ---------- 내부 유틸 ----------

    def _normalize_date(self, date):
        if date is None or "오늘" in str(date):
            return _date.today().isoformat()
        return date

    def _add_record(self, date, tag, amount, action, id):
        self.records.append({"date": date, "tag": tag, "amount": amount, "action": action, "id": id})

    def _current_amount(self, id):
        return sum(record["amount"] for record in self.records if record["id"] == id)

    def _aggregate_nonzero(self):
        totals = {}
        for record in self.records:
            id = record["id"]
            if id not in totals:
                totals[id] = {"date": record["date"], "tag": record["tag"], "amount": 0}
            totals[id]["amount"] += record["amount"]

        return [
            {"id": id, "date": total["date"], "tag": total["tag"], "amount": total["amount"]}
            for id, total in totals.items()
            if total["amount"] != 0
        ]

    def _pad_date(self, d, end):
        if d and len(d) == 7:  # 'YYYY-MM' -> 월의 시작/끝일로 확장
            return d + ("-31" if end else "-01")
        return d

    # ---------- 공개 API ----------

    def register(self, amount, date=None, tag=None):
        date = self._normalize_date(date)
        tag = tag or "기타"
        self._next_id += 1
        self._add_record(date, tag, amount, "등록", id=self._next_id)
        return {"ok": True, "id": self._next_id, "date": date, "tag": tag, "amount": amount}

    def alter(self, date=None, tag=None, new_amount=None, id=None):
        """id가 없으면 date/tag로 search해서 후보를 찾아 ok:False로 돌려줌 (실행은 id로만 함)."""
        if id:
            current = self._current_amount(id)
            if current == 0:
                return {"ok": False, "message": f"{id}에 해당하는 {self.label} 내역이 없습니다."}

            data = next(record for record in self.records if record["id"] == id)
            date, tag = data["date"], data["tag"]

            self._add_record(date, tag, -current, "수정", id)
            self._add_record(date, tag, new_amount, "수정", id)
            return {"ok": True, "id": id, "date": date, "tag": tag, "before": current, "after": new_amount}

        result = self.search(s_date=date, e_date=date, tag=tag)
        result["ok"] = False
        return result

    def delete(self, date=None, tag=None, id=None):
        """id가 없으면 date/tag로 search해서 후보를 찾아 ok:False로 돌려줌 (실행은 id로만 함)."""
        if id:
            current = self._current_amount(id)
            if current == 0:
                return {"ok": False, "message": f"{id}에 해당하는 {self.label} 내역이 없습니다."}

            data = next(record for record in self.records if record["id"] == id)
            date, tag = data["date"], data["tag"]

            self._add_record(date, tag, -current, "삭제", id)
            return {"ok": True, "id": id, "date": date, "tag": tag, "deleted_amount": current}

        result = self.search(s_date=date, e_date=date, tag=tag)
        result["ok"] = False
        return result

    def search(self, s_date=None, e_date=None, tag=None):
        if not s_date and not e_date and not tag:
            return {"ok": False, "message": "날짜 또는 태그 중 하나는 입력해야 합니다."}

        candidates = self._aggregate_nonzero()
        if s_date:
            candidates = [c for c in candidates if c["date"] >= self._pad_date(s_date, end=False)]
        if e_date:
            candidates = [c for c in candidates if c["date"] <= self._pad_date(e_date, end=True)]

        if not tag:
            if candidates:
                return {"ok": True, "items": candidates}
            return {"ok": False, "message": f"해당 기간의 {self.label} 내역이 없습니다."}

        exact = [c for c in candidates if c["tag"] == tag]
        if exact:
            return {"ok": True, "items": exact}
        if candidates:
            return {
                "ok": True,
                "items": candidates,
                "note": "정확히 일치하는 태그가 없어 해당 기간 전체 내역을 반환합니다.",
            }
        return {"ok": False, "message": f"해당 조건에 맞는 {self.label} 내역이 없습니다."}


# def demo():
#     budget = Ledger("예산")
#     expenditure = Ledger("지출")

#     r1 = budget.register(1000000, "2026-08-01", "생활비")
#     assert r1 == {"ok": True, "id": 1, "date": "2026-08-01", "tag": "생활비", "amount": 1000000}

#     # 같은 날 같은 태그로 두 번 지출 -> id로 구분되어야 함 (자연키 충돌 회귀 테스트)
#     e1 = expenditure.register(13000, "2026-08-18", "카페")
#     e2 = expenditure.register(5000, "2026-08-18", "카페")
#     assert e1["id"] == 1 and e2["id"] == 2

#     s = expenditure.search(s_date="2026-08-18", e_date="2026-08-18", tag="카페")
#     assert s["ok"] is True and len(s["items"]) == 2

#     d = expenditure.delete(id=e2["id"])
#     assert d == {"ok": True, "id": 2, "date": "2026-08-18", "tag": "카페", "deleted_amount": 5000}
#     assert all(rec["date"] is not None and rec["tag"] is not None for rec in expenditure.records)

#     a = expenditure.alter(id=e1["id"], new_amount=9000)
#     assert a["ok"] is True and a["before"] == 13000 and a["after"] == 9000
#     assert expenditure._current_amount(e1["id"]) == 9000

#     # 존재하지 않거나 이미 삭제된 id
#     assert expenditure.delete(id=999)["ok"] is False
#     assert expenditure.delete(id=e2["id"])["ok"] is False

#     # id 없이 date/tag만 -> 후보 1개여도 항상 ok:False (실행은 반드시 id로)
#     no_id = expenditure.alter(date="2026-08-18", tag="카페", new_amount=1)
#     assert no_id["ok"] is False and no_id["items"][0]["id"] == e1["id"]

#     # label이 에러 메시지에 반영되는지 확인
#     missing = budget.delete(id=999)
#     assert missing == {"ok": False, "message": "999에 해당하는 예산 내역이 없습니다."}

#     # 태그 오타 -> 정확히 없으면 해당 기간 전체 반환
#     fuzzy = expenditure.search(s_date="2026-08", e_date="2026-08", tag="카펭")
#     assert fuzzy["ok"] is True and "note" in fuzzy

#     # 존재하지 않는 기간
#     none_found = budget.search(s_date="2099-01", e_date="2099-01")
#     assert none_found == {"ok": False, "message": "해당 기간의 예산 내역이 없습니다."}

#     print("demo ok")


# demo()


### 2. 예산 지출 거래 내역 보고서 저장
- 남은 예산 확인(예산-지출)
- 예산 지출 내역 월별 보고서 마크다운 파일 만들기
- 폴더 생성하고(reports) 그 곳에 파일 저장

In [7]:
"""예산 대비 지출 리포트."""

import os

from ledger import Ledger

REPORT_DIR = os.path.join(os.getcwd(), "reports")


def remaining(budget: Ledger, expenditure: Ledger, month: str) -> int:
    """month는 'YYYY-MM' 형식. 해당 월의 예산 총합 - 지출 총합."""
    total_budget = sum(r["amount"] for r in budget.records if r["date"].startswith(month))
    total_spent = sum(r["amount"] for r in expenditure.records if r["date"].startswith(month))
    return total_budget - total_spent


def _items(records_owner, month):
    result = records_owner.search(s_date=month, e_date=month)
    return sorted(result["items"], key=lambda it: it["date"]) if result["ok"] else []


def _table(items):
    if not items:
        return "내역 없음\n"
    lines = ["| 날짜 | 태그 | 금액 |", "|---|---|---|"]
    lines += [f"| {it['date']} | {it['tag']} | {it['amount']:,}원 |" for it in items]
    return "\n".join(lines) + "\n"


def generate_monthly_report(budget: Ledger, expenditure: Ledger, month: str) -> str:
    """month는 'YYYY-MM' 형식. 해당 월의 예산/지출 내역을 Markdown 문자열로 정리."""
    budget_items = _items(budget, month)
    expenditure_items = _items(expenditure, month)
    total_budget = sum(it["amount"] for it in budget_items)
    total_spent = sum(it["amount"] for it in expenditure_items)

    return (
        f"# {month} 거래 내역\n\n"
        f"## 예산\n{_table(budget_items)}\n"
        f"**총 예산: {total_budget:,}원**\n\n"
        f"## 지출\n{_table(expenditure_items)}\n"
        f"**총 지출: {total_spent:,}원**\n\n"
        f"## 요약\n남은 예산: {total_budget - total_spent:,}원\n"
    )


def save_monthly_report(budget: Ledger, expenditure: Ledger, month: str) -> str:
    """리포트를 REPORT_DIR/{month}.md로 저장하고 경로를 반환."""
    os.makedirs(REPORT_DIR, exist_ok=True)
    path = os.path.join(REPORT_DIR, f"{month}.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(generate_monthly_report(budget, expenditure, month))
    return path


# def demo():
#     b = Ledger("예산")
#     b.register(1000000, "2026-08-01", "생활비")

#     e = Ledger("지출")
#     e.register(30000, "2026-08-05", "케익점")
#     e.register(60000, "2026-08-10", "나이키")

#     assert remaining(b, e, "2026-08") == 1000000 - 90000
#     assert remaining(b, e, "2026-07") == 0  # 해당 월 기록 없음

#     md = generate_monthly_report(b, e, "2026-08")
#     assert "케익점" in md and "나이키" in md and "남은 예산: 910,000원" in md

#     empty_md = generate_monthly_report(b, e, "2026-07")
#     assert "내역 없음" in empty_md

#     path = save_monthly_report(b, e, "2026-08")
#     assert os.path.exists(path)
#     with open(path, encoding="utf-8") as f:
#         assert f.read() == md
#     os.remove(path)

#     print("demo ok")



# demo()


### 3. 도구 스키마와 Function calling이 호출할 함수 정의
- 예산 및 지출 등록/수정/삭제/검색
- system_instruction 정의
- 모델이 보낸 function_call step을 처리
- 예산 및 지출 기록 save/load 로직 정의
    - 거래 기록
    - 다음에 발급할 id 번호

In [8]:
"""챗봇(Gemini function calling)이 호출할 도구 스키마 + 디스패처."""

import json
import os
from datetime import date as _date

from ledger import Ledger
from report import remaining, save_monthly_report

# 세션 동안 유지되는 공유 상태 (지금은 인메모리 싱글턴)
budget = Ledger("예산")
expenditure = Ledger("지출")


# 1) 실제 로직
def get_remaining_budget(month: str = None) -> dict:
    month = month or _date.today().isoformat()[:7]
    amount = remaining(budget, expenditure, month)
    return {"ok": True, "month": month, "remaining": amount}


def register_expenditure(amount: int, date: str = None, tag: str = None) -> dict:
    return expenditure.register(amount, date, tag)


def alter_expenditure(date: str = None, tag: str = None, new_amount: int = None, id: int = None) -> dict:
    return expenditure.alter(date, tag, new_amount, id)


def delete_expenditure(date: str = None, tag: str = None, id: int = None) -> dict:
    return expenditure.delete(date, tag, id)

def search_expenditure(s_date: str = None, e_date: str = None, tag: str = None) -> dict:
    if not s_date and not e_date and not tag:
        s_date = e_date = _date.today().isoformat()[:7]
    return expenditure.search(s_date, e_date, tag)

def register_budget(amount: int, date: str = None, tag: str = None) -> dict:
    return budget.register(amount, date, tag)


def alter_budget(date: str = None, tag: str = None, new_amount: int = None, id: int = None) -> dict:
    return budget.alter(date, tag, new_amount, id)


def delete_budget(date: str = None, tag: str = None, id: int = None) -> dict:
    return budget.delete(date, tag, id)

def search_budget(s_date: str = None, e_date: str = None, tag: str = None) -> dict:
    if not s_date and not e_date and not tag:
        s_date = e_date = _date.today().isoformat()[:7]
    return budget.search(s_date, e_date, tag)


def save_report(month: str = None) -> dict:
    month = month or _date.today().isoformat()[:7]
    path = save_monthly_report(budget, expenditure, month)
    return {"ok": True, "month": month, "path": path}


# 2) 도구 스키마 (Gemini API function calling 형식)
get_remaining_budget_tool = {
    "type": "function",
    "name": "get_remaining_budget",
    "description": "특정 월의 예산에서 지출을 뺀 남은 금액을 조회합니다. month를 안 주면 이번 달 기준.",
    "parameters": {
        "type": "object",
        "properties": {
            "month": {"type": "string", "description": "조회할 월. 형식 'YYYY-MM'. 생략 시 이번 달."},
        },
        "required": [],
    },
}

_AMOUNT_DATE_TAG_PROPS = {
    "amount": {"type": "integer", "description": "금액(원)"},
    "date": {"type": "string", "description": "날짜 'YYYY-MM-DD'. 생략하거나 '오늘' 포함 시 오늘 날짜."},
    "tag": {"type": "string", "description": "카테고리 태그. 생략 시 '기타'."},
}

_DATE_TAG_NEW_AMOUNT_PROPS = {
    "id": {"type": "integer", "description": "수정할 거래의 id. 이게 있으면 date/tag는 무시하고 바로 실행합니다."},
    "date": {"type": "string", "description": "id 없을 때 후보를 찾기 위한 날짜 'YYYY-MM-DD'."},
    "tag": {"type": "string", "description": "id 없을 때 후보를 찾기 위한 태그."},
    "new_amount": {"type": "integer", "description": "수정 후 금액(원)."},
}

_DATE_TAG_PROPS = {
    "id": {"type": "integer", "description": "삭제할 거래의 id. 이게 있으면 date/tag는 무시하고 바로 실행합니다."},
    "date": {"type": "string", "description": "id 없을 때 후보를 찾기 위한 날짜 'YYYY-MM-DD'."},
    "tag": {"type": "string", "description": "id 없을 때 후보를 찾기 위한 태그."},
}

_DATE_TAG_SEARCH_PROPS = {
    "s_date": {"type": "string", "description": "검색 시작 날짜 'YYYY-MM-DD' 또는 'YYYY-MM'. 생략 가능."},
    "e_date": {"type": "string", "description": "검색 마지막 날짜 'YYYY-MM-DD' 또는 'YYYY-MM'. 생략 가능."},
    "tag": {"type": "string", "description": "검색할 태그. 생략 가능."},
}

register_expenditure_tool = {
    "type": "function",
    "name": "register_expenditure",
    "description": "지출 내역을 등록합니다.",
    "parameters": {"type": "object", "properties": _AMOUNT_DATE_TAG_PROPS, "required": ["amount"]},
}

alter_expenditure_tool = {
    "type": "function",
    "name": "alter_expenditure",
    "description": "지출 내역의 금액을 수정합니다. id로만 실제 실행되며, id 없이 date/tag만 주면 후보 목록만 반환합니다.",
    "parameters": {"type": "object", "properties": _DATE_TAG_NEW_AMOUNT_PROPS, "required": []},
}

delete_expenditure_tool = {
    "type": "function",
    "name": "delete_expenditure",
    "description": "지출 내역을 삭제합니다. id로만 실제 실행되며, id 없이 date/tag만 주면 후보 목록만 반환합니다.",
    "parameters": {"type": "object", "properties": _DATE_TAG_PROPS, "required": []},
}

search_expenditure_tool = {
    "type": "function",
    "name": "search_expenditure",
    "description": "s_date~e_date 기간 및/또는 태그로 지출 내역을 검색합니다. 아무것도 안 주면 이번 달 기준으로 반환합니다.",
    "parameters": {"type": "object", "properties": _DATE_TAG_SEARCH_PROPS, "required": []},
}

register_budget_tool = {
    "type": "function",
    "name": "register_budget",
    "description": "예산을 등록합니다.",
    "parameters": {"type": "object", "properties": _AMOUNT_DATE_TAG_PROPS, "required": ["amount"]},
}

alter_budget_tool = {
    "type": "function",
    "name": "alter_budget",
    "description": "예산의 금액을 수정합니다. id로만 실제 실행되며, id 없이 date/tag만 주면 후보 목록만 반환합니다.",
    "parameters": {"type": "object", "properties": _DATE_TAG_NEW_AMOUNT_PROPS, "required": []},
}

delete_budget_tool = {
    "type": "function",
    "name": "delete_budget",
    "description": "예산을 삭제합니다. id로만 실제 실행되며, id 없이 date/tag만 주면 후보 목록만 반환합니다.",
    "parameters": {"type": "object", "properties": _DATE_TAG_PROPS, "required": []},
}

search_budget_tool = {
    "type": "function",
    "name": "search_budget",
    "description": "날짜+태그로 특정된 예산을 검색합니다. date/tag가 정확하지 않으면 이번 달을 기준으로 반환합니다.",
    "parameters": {"type": "object", "properties": _DATE_TAG_SEARCH_PROPS, "required": []},
}

save_report_tool = {
    "type": "function",
    "name": "save_report",
    "description": "특정 월의 예산/지출 내역을 Markdown 파일로 저장합니다. month를 안 주면 이번 달 기준.",
    "parameters": {
        "type": "object",
        "properties": {
            "month": {"type": "string", "description": "저장할 월. 형식 'YYYY-MM'. 생략 시 이번 달."},
        },
        "required": [],
    },
}

# 3) Gemini에 넘길 스키마 목록
TOOLS = [
    get_remaining_budget_tool,
    register_expenditure_tool,
    alter_expenditure_tool,
    delete_expenditure_tool,
    search_expenditure_tool,
    register_budget_tool,
    alter_budget_tool,
    delete_budget_tool,
    search_budget_tool,
    save_report_tool,
]

# 4) 등록: 도구 이름 -> 함수 (키는 각 스키마의 "name" 값과 동일해야 함)
TOOL_FUNCTIONS = {
    "get_remaining_budget": get_remaining_budget,
    "register_expenditure": register_expenditure,
    "alter_expenditure": alter_expenditure,
    "delete_expenditure": delete_expenditure,
    "search_expenditure": search_expenditure,
    "register_budget": register_budget,
    "alter_budget": alter_budget,
    "delete_budget": delete_budget,
    "search_budget": search_budget,
    "save_report": save_report,
}


# 5) 시스템 지침 (Gemini agent에 그대로 전달)
system_instruction = f"""
등록, 수정, 삭제를 실행하기 전에 한 번 더 사용자에게 확인 질문을 해야 합니다. 반드시.
오늘 날짜는 {_date.today().isoformat()}입니다. 사용자가 연도를 생략하면 이 날짜를 기준으로 추론하세요.
alter/delete는 id로만 실제로 실행됩니다. id 없이 date/tag만 주면 항상 ok:false와 후보 목록(items)만 돌아옵니다.
items를 사용자에게 보여주고 어떤 항목을 말하는지 재질문한 뒤, 사용자가 고른 항목의 id로 alter/delete를 다시 호출하세요.
register 응답에도 id가 들어있으니, 방금 등록한 거래를 바로 수정/삭제해야 하면 그 id를 재사용하세요. 임의로 하나를 골라 실행하지 마세요.
새 지출/예산을 등록하기 전에 search로 기존 태그를 확인하고, 비슷한 태그가 있으면 새로 만들지 말고 그 태그를 재사용하세요.
여기 정의된 도구로 할 수 없는 요청(통계, 그래프 등)은 지어내지 말고 아직 지원하지 않는다고 답하세요.
금액이 음수이거나 비정상적으로 크면 그대로 등록하지 말고 사용자에게 다시 확인하세요.
함수 호출 결과(raw dict)를 그대로 보여주지 말고, 사람이 읽기 쉬운 자연어 문장으로 정리해서 답하세요.
사용자가 종료하거나 끝내고 싶다고 하면 quit or exit 를 입력하도록 유도하세요.
"""


# 6) 실행 디스패처: 모델이 보낸 function_call step을 처리
def execute_tool_call(name: str, arguments: dict) -> dict:
    func = TOOL_FUNCTIONS.get(name)
    if func is None:
        return {"ok": False, "error": f"허용되지 않은 도구: {name}"}
    return func(**arguments)


# 7) 영속화: budget/expenditure의 records를 JSON 파일로 저장/복원
STATE_FILE = os.path.join(os.getcwd(), "data.json")


def save_state(path=STATE_FILE):
    state = {
        "budget": {"records": budget.records, "next_id": budget._next_id},
        "expenditure": {"records": expenditure.records, "next_id": expenditure._next_id},
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def load_state(path=STATE_FILE):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    budget.records = data["budget"]["records"]
    budget._next_id = data["budget"]["next_id"]
    expenditure.records = data["expenditure"]["records"]
    expenditure._next_id = data["expenditure"]["next_id"]


# def demo():
#     today_month = _date.today().isoformat()[:7]
#     budget_reg = execute_tool_call(
#         "register_budget", {"amount": 1000000, "date": f"{today_month}-01", "tag": "생활비"}
#     )
#     expenditure_reg = execute_tool_call(
#         "register_expenditure", {"amount": 300000, "date": f"{today_month}-05", "tag": "식비"}
#     )

#     r = execute_tool_call("get_remaining_budget", {})
#     assert r == {"ok": True, "month": today_month, "remaining": 700000}

#     r2 = execute_tool_call("get_remaining_budget", {"month": "2099-01"})
#     assert r2 == {"ok": True, "month": "2099-01", "remaining": 0}

#     # id 없이 date/tag만 주면 실행되지 않고 후보만 반환
#     no_id = execute_tool_call(
#         "alter_expenditure", {"date": f"{today_month}-05", "tag": "식비", "new_amount": 100000}
#     )
#     assert no_id["ok"] is False and no_id["items"][0]["id"] == expenditure_reg["id"]

#     # id로 실제 실행
#     a = execute_tool_call(
#         "alter_expenditure", {"id": expenditure_reg["id"], "new_amount": 100000}
#     )
#     assert a["ok"] is True and a["before"] == 300000 and a["after"] == 100000
#     assert execute_tool_call("get_remaining_budget", {})["remaining"] == 900000

#     d = execute_tool_call("delete_budget", {"id": budget_reg["id"]})
#     assert d["ok"] is True and d["deleted_amount"] == 1000000
#     assert execute_tool_call("get_remaining_budget", {})["remaining"] == -100000

#     s = execute_tool_call("search_expenditure", {"tag": "식비"})
#     assert s["ok"] is True and s["items"][0]["amount"] == 100000

#     s2 = execute_tool_call("search_expenditure", {})  # 아무것도 안 주면 이번 달 기준
#     assert s2["ok"] is True

#     rpt = execute_tool_call("save_report", {"month": today_month})
#     assert rpt["ok"] is True and os.path.exists(rpt["path"])
#     os.remove(rpt["path"])

#     assert execute_tool_call("nope", {}) == {"ok": False, "error": "허용되지 않은 도구: nope"}

#     # 영속화: _next_id도 같이 저장/복원되는지 확인
#     save_state()
#     saved_next_id = expenditure._next_id
#     expenditure.records = []
#     expenditure._next_id = 0
#     load_state()
#     assert expenditure._next_id == saved_next_id
#     assert execute_tool_call("search_expenditure", {"tag": "식비"})["items"][0]["amount"] == 100000
#     os.remove(STATE_FILE)

#     print("demo ok")



# demo()


### 4. main - agent 가동하기
- Gemini API 연결
- 이전 데이터 불러오기 (test 단계로 이전 데이터 없으면 seed 데이터 가지고 함)
- Gemini 채팅봇 실행

In [10]:
"""Gemini function calling 챗봇 루프. tools.py의 TOOLS/system_instruction/execute_tool_call을 그대로 사용."""

import json
import os

from dotenv import load_dotenv
from google import genai

from tools import STATE_FILE, TOOLS, execute_tool_call, load_state, save_state, system_instruction

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정하세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(api_key=api_key)


def run_agent(user_input: str, previous_interaction_id: str = None, max_turns: int = 5) -> dict:
    next_input = user_input
    logs = []

    for turn in range(1, max_turns + 1):
        request = {
            "model": model,
            "input": next_input,
            "system_instruction": system_instruction,
            "tools": TOOLS,
            "store": True,
        }
        if previous_interaction_id is not None:
            request["previous_interaction_id"] = previous_interaction_id

        interaction = client.interactions.create(**request)
        function_calls = [step for step in interaction.steps if step.type == "function_call"]

        if not function_calls:
            return {
                "ok": True,
                "answer": interaction.output_text,
                "turns": turn,
                "tool_logs": logs,
                "previous_interaction_id": interaction.id,
            }

        next_input = []
        for step in function_calls:
            result = execute_tool_call(step.name, step.arguments)
            save_state()  # 등록/수정/삭제 즉시 반영, 다음 실행에도 남도록
            logs.append({"turn": turn, "tool": step.name, "arguments": step.arguments, "result": result})
            next_input.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}],
            })

        previous_interaction_id = interaction.id

    return {
        "ok": False,
        "answer": None,
        "turns": max_turns,
        "tool_logs": logs,
        "error": "최대 반복 횟수를 초과했습니다.",
        "previous_interaction_id": previous_interaction_id,
    }



if os.path.exists(STATE_FILE):
    load_state()
    print("이전에 저장된 데이터를 불러왔습니다.")
else:
    from seed_data import seed

    print("저장된 데이터가 없어 샘플 데이터로 시작합니다:")
    for line in seed():
        print(" ", line)
    save_state()

previous_interaction_id = None
print("\n예산 관리 챗봇입니다. 종료하려면 'quit' 입력.")
while True:
    user_input = input("\n나: ")
    if user_input.strip().lower() in ("quit", "exit"):
        break
    result = run_agent(user_input, previous_interaction_id=previous_interaction_id)
    previous_interaction_id = result["previous_interaction_id"]
    print("챗봇:", result["answer"] if result["ok"] else result["error"])


저장된 데이터가 없어 샘플 데이터로 시작합니다:
  [예산] 2025-01-01 생활비 900,000원
  [예산] 2025-06-01 생활비 950,000원
  [예산] 2025-12-01 생활비 1,000,000원
  [예산] 2026-01-01 생활비 1,000,000원
  [예산] 2026-03-01 생활비 1,000,000원
  [예산] 2026-06-01 생활비 1,000,000원
  [예산] 2026-08-01 생활비 1,000,000원
  [예산] 2026-08-01 문화 100,000원
  [지출] 2025-01-15 식비 45,000원
  [지출] 2025-01-22 교통 30,000원
  [지출] 2025-06-05 카페 12,000원
  [지출] 2025-06-20 쇼핑 80,000원
  [지출] 2025-12-24 경조사 100,000원
  [지출] 2026-01-10 식비 320,000원
  [지출] 2026-01-25 통신비 55,000원
  [지출] 2026-03-14 카페 15,000원
  [지출] 2026-06-02 교통 40,000원
  [지출] 2026-08-05 식비 320,000원
  [지출] 2026-08-10 카페 45,000원
  [지출] 2026-08-18 카페 13,000원
  [지출] 2026-08-18 카페 5,000원
  [지출] 2026-08-20 교통 60,000원
  [지출] 2026-08-25 문화 30,000원

예산 관리 챗봇입니다. 종료하려면 'quit' 입력.
챗봇: 2026년 8월 보고서가 성공적으로 생성 및 저장되었습니다! 

저장된 파일 경로: `c:\Users\user\aim-ai-my\python\mini_project\reports\2026-08.md`
챗봇: 2026년 8월 카페 지출 내역은 다음과 같습니다:

- **8월 10일**: 45,000원
- **8월 18일**: 13,000원
- **8월 18일**: 5,000원

총 3건의 지출이 있으며, 합계는 **63,000원**